# HartmannTwoWay — the driven Hartmann flow with both feedbacks live

`MagneticCoupled` · ASN 28 validation case (TRUST-LiMe).


## Introduction

The Hartmann two-way follow-up announced by `Coupled_Feedback`: both
feedbacks of the two-way coupling are live **and** the check is quantitative.
A channel between plates at $y = \pm h$ ($h = 0.5$) is driven by an imposed
pressure difference $\Delta p = 1$ over $L = 1$ (open boundaries at
$x = 0, 1$). The magnetic equation transports the **solved** NS velocity
(`vitesse_hydraulique`) and carries the M1 `induction_stretching` source;
the momentum equation carries the `force_lorentz` source. With $B_0 = 1$
along $y$, $\eta = 0.01$, $\mu = \mu_0 = \rho = 1$:

$$ \mathrm{Ha} = \frac{B_0 h}{\sqrt{\mu \mu_0 \eta}} = 5 $$

and the coupled steady state is **exactly the classical Hartmann solution**,
$$ u(y) = \frac{G h^2}{\mu \mathrm{Ha}^2}\left[1 - \frac{\cosh(\mathrm{Ha}\,y/h)}{\cosh(\mathrm{Ha})}\right],
\qquad
b_x(y) = \frac{B_0}{\eta}\frac{G h^2}{\mu \mathrm{Ha}^2}\left[\frac{h}{\mathrm{Ha}\cosh(\mathrm{Ha})}\sinh\!\left(\frac{\mathrm{Ha}\,y}{h}\right) - y\right], $$
with $G = \Delta p/L$: an even, flattened velocity profile and an odd,
mean-free induced field ($b_x(0) = 0$) generated by the **solved** velocity
— the full two-way loop $u \to$ stretching $\to b_x \to$ Lorentz force
$\to u$ is exercised.

**Two-way discriminator.** Without the Lorentz feedback the flow would be
Poiseuille, $u_P(y) = G(h^2 - y^2)/(2\mu)$, whose centreline velocity is
$\cosh(\mathrm{Ha})(\cosh(\mathrm{Ha})-1)^{-1} \approx 12.7$ times the
Hartmann one. The measured profile must sit orders of magnitude closer to
Hartmann than to Poiseuille — a sharp, quantitative test of the feedback
(no band like the historical $10^{-3} < \|b - b_0\| < 0.5$ discriminator).

**Magnetic-pressure boundary profiles.** The $y$-momentum balance closes
with $p(y) + b_x(y)^2/(2\mu_0)$ uniform in $y$, so the open pressure
boundaries impose the analytic profile $p(y) = P - b_x(y)^2/(2\mu_0)$
(`Champ_front_fonc_txyz`). A *uniform* imposed pressure would conflict with
the magnetic pressure, drive a spurious cross-flow ($u_y \sim 5\times10^{-4}$,
measured) and pollute $b_y$ through the stretching source — measured drift
of $b_y$: **9.6 %** with uniform $p$, **0.75 %** with the profile. The
residual cross-flow is checked directly below.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 1 | $b_y = B_0$ near-exact invariant ($S_y = 0$) | $\max_t \|b_y - B_0\| < 2\times10^{-2}$ |
| 2 | two-way: solved $u$ relaxes to the Hartmann profile | core $\|u - u_H\|/U_0 < 1\%$, stationary |
| 3 | Lorentz feedback suppresses the flow (not Poiseuille) | $\|u - u_H\| \ll \|u_P - u_H\|$ (ratio $< 5\%$) |
| 4 | stretching from the solved $u$ induces $b_x$ | core $\|b_x - b_x^{ref}\|/B_{scale} < 1\%$ |
| 5 | magnetic-pressure boundary profile kills the cross-flow | $\|u_y\| < 10^{-4}$ |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: analytic Hartmann reference and the `.son` probe reader

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and read back from
the build directory. The `vitesse` probes record both components
($(u_x, u_y)$ — $u_y$ is the cross-flow residual).


In [ ]:
import math
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name, sub=None):
    """Return (time, values) for probe `name` of case `case` (dataset name).
    Optional `sub` selects a build subdirectory (e.g. 'hr', 'ha_2.5')."""
    if sub is not None:
        cands = [f"{BUILD}/{sub}/{case}_{name}.son"]
    else:
        cands = [f"{BUILD}/coarse/{case}_{name}.son",
                 f"{BUILD}/{case}_{name}.son"]
        if case.startswith("PAR_"):
            cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[0])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_comps(case, name, sub=None):
    """All component series of a (possibly multi-component) probe.

    A `vitesse` probe records one entry per component; entries[0] is u_x,
    entries[1] is u_y (each a 1-D time series). Optional `sub` selects a
    build subdirectory."""
    if sub is not None:
        cands = [f"{BUILD}/{sub}/{case}_{name}.son"]
    else:
        cands = [f"{BUILD}/coarse/{case}_{name}.son",
                 f"{BUILD}/{case}_{name}.son"]
        if case.startswith("PAR_"):
            cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[0])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    return [np.asarray(son.getValues(e)[1], dtype=float) for e in entries]

# --- analytic coupled Hartmann reference (same constants as the dataset) ---
Ha, H, G, MU, ETA, B0, MU0, RHO = 5.0, 0.5, 1.0, 1.0, 0.01, 1.0, 1.0, 1.0
CH = math.cosh(Ha)
C   = G*H*H/(MU*Ha*Ha)                 # = 0.01, amplitude factor of u(y)
U0c = C*(1 - 1/CH)                    # centreline velocity (Hartmann)
B_SCALE = (B0/ETA)*C*(H - math.sinh(Ha)*H/(Ha*CH))   # ~ wall value of b_x

def u_hartmann(y):
    """Classical Hartmann profile (the exact coupled steady solution)."""
    return C*(1 - math.cosh(Ha*y/H)/CH)

def u_poiseuille(y):
    """The no-Lorentz-force profile: the two-way discriminator's other pole."""
    return (G/(2*MU))*(H*H - y*y)

def b_x_steady(y):
    """Mean-free induced field of the coupled steady state (odd profile)."""
    return (B0/ETA)*C*(math.sinh(Ha*y/H)*H/(Ha*CH) - y)

# probes sit exactly on cell centres: y_k = -0.5 + (k+1/2)/64, x = 0.375
CORE = [("SONDE_U0", 0.0078125), ("SONDE_U1", -0.0546875),
        ("SONDE_U2", -0.1796875), ("SONDE_U3", -0.3046875)]
WALL = [("SONDE_UC", -0.4453125)]
CORE_B = [("SONDE_B0", 0.0078125), ("SONDE_B1", -0.0546875),
          ("SONDE_B2", -0.1796875), ("SONDE_B3", -0.3046875)]

print("BUILD =", BUILD)
print(f"U0 (centreline, Hartmann) = {U0c:.6e} ; Poiseuille centreline = {G*H*H/(2*MU):.6e}")
print(f"B_SCALE (induced field amplitude) = {B_SCALE:.6e}")


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/hartmann_two_way.data carries $nx/$ny node-count
# placeholders; every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 in the active y direction
#   fine     = cells x4 in the active y direction
# Keep x resolution fixed so every scale samples the same streamwise cell.
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NY0 = 65                              # baseline y node count (coarse)
meshes = {
    "coarse":   {"nx": 5, "ny": NY0},
    "standard": {"nx": 5, "ny": 2*NY0-1},
    "fine":     {"nx": 5, "ny": 4*NY0-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("hartmann_two_way.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition hartmann_two_way.data 2 -> PAR_hartmann_two_way.data + DOM_000*.Zones

# --- Hartmann-number scan (regime map) --------------------------------------
# The same form at Ha = 2.5 and 7.9 (Ha = 5 is the mesh-scan baseline above).
# eta = (B0 h/Ha)^2/(mu mu0) = 0.25/Ha^2; tmax = 1.3 x h^2/eta (the magnetic
# diffusion time, so every level reaches its steady state).
for ha in (2.5, 7.9):
    eta = 0.25/ha**2
    dic = {"nx": 5, "ny": 65, "eta": repr(eta), "ka": repr(ha/0.5),
           "kfac": repr(0.5/(ha*math.cosh(ha))),
           "tmax": repr(math.ceil(0.25/eta*1.3*10)/10)}
    cases[f"ha_{ha}"] = run.addCaseFromTemplate("hartmann_two_way_ha.data", f"ha_{ha}", dic=dic,
                                                nbProcs=NB_PROCS, excluNR=True)
    cases[f"ha_{ha}"].partition()
# --- High-Rm case (G = 2, Rm = 2) at Ha = 5 ------------------------------
# Single (non-scan) case at higher driving pressure / Rm to test the
# lagged coupling beyond the Rm = 1 identity of the Ha scan.
hr_case = run.addCaseFromTemplate("hartmann_two_way_highrm.data", "hr",
                                 dic={"nx": 5, "ny": 65}, nbProcs=NB_PROCS, excluNR=True)
hr_case.partition()

# --- full Re_H x Ha map ------------------------------------------------------
# Re_H = rho * U_H(0) * L / mu, with L=1 and U_H(0) the Hartmann centreline
# speed. Vary mu for Re, eta to hold Rm=0.1, and B0 to hold Ha. Each pair has
# its own build directory because partition() writes DOM_000*.Zones files.
# Use ten diffusion times to let the staggered transient reach steady state.
RE_VALUES = (0.01, 0.1, 1.0, 10.0)
HA_VALUES = (2.5, 5.0, 7.9)
RM_TARGET = 0.1
H_MAP, RHO_MAP, MU0_MAP, L_MAP, G_MAP = 0.5, 1.0, 1.0, 1.0, 1.0
for re_target in RE_VALUES:
    for ha in HA_VALUES:
        centre_shape = 1.0 - 1.0 / math.cosh(ha)
        mu = math.sqrt(RHO_MAP*G_MAP*H_MAP**2*L_MAP*centre_shape /
                       (re_target*ha**2))
        u_center = G_MAP*H_MAP**2/(mu*ha**2)*centre_shape
        eta = u_center*L_MAP/RM_TARGET
        b0 = ha*math.sqrt(mu*MU0_MAP*eta)/H_MAP
        C_reha = G_MAP*H_MAP**2/(mu*ha**2)
        bscale = b0/eta*C_reha
        pcoeff = bscale**2/(2.0*MU0_MAP)
        tscale = max(H_MAP**2/eta, RHO_MAP*H_MAP**2/mu)
        # The intermediate-Re, high-Ha corner has a slow staggered transient;
        # reduce dt there and allow more settling time. The high-Re/high-Ha
        # point gets a modestly longer run to meet the same stationarity gate.
        settling_factor = 40.0 if re_target == 0.1 else 10.0
        dt = 0.01 if re_target == 0.1 else 0.05
        sub = f"re_{re_target:g}_ha_{ha:g}"
        dic = {"nx": 5, "ny": 65, "mu": repr(mu), "eta": repr(eta),
               "B0": repr(b0), "ka": repr(ha/H_MAP),
               "kfac": repr(H_MAP/(ha*math.cosh(ha))),
               "bscale": repr(bscale), "pcoeff": repr(pcoeff), "dt": repr(dt),
               "tmax": repr(math.ceil(tscale*settling_factor*10)/10)}
        cases[sub] = run.addCaseFromTemplate("hartmann_two_way_reha.data", sub,
                                             dic=dic, nbProcs=NB_PROCS,
                                             excluNR=True)
        cases[sub].partition()

run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks


In [ ]:
# 1) the applied transverse field is a near-exact invariant
#    (residual: the discrete magnetic-pressure balance at the open boundaries;
#    a UNIFORM imposed p would leave a 9.6% drift - see the introduction)
TOL_BY = 2e-2
t, by = read_son(case.dataFileName_, "SONDE_BY")
drift_by = float(np.max(np.abs(by - B0)))
print(f"b_y: max |b_y - B0| = {drift_by:.2e}  (bound {TOL_BY:.0e})")
assert drift_by < TOL_BY, f"applied field drifted: {drift_by:.3e}"
print("OK: b_y = B0 is a near-exact invariant of the coupled steady state.")


In [ ]:
# 2) the solved velocity relaxes to the analytic Hartmann profile (two-way)
TOL_U = 1e-2
STATIONARY = 1e-4
core_err = []
for name, y in CORE:
    ux = read_son_comps(case.dataFileName_, name)[0]
    ref = u_hartmann(y)
    rel = abs(ux[-1] - ref) / abs(ref)
    step = abs(ux[-1] - ux[-2]) / U0c
    core_err.append(rel)
    print(f"{name}  y = {y:+.7f} : u_x = {ux[-1]:.6e}  (Hartmann {ref:.6e})  "
          f"rel = {rel:.2e}  last-step/U0 = {step:.2e}")
worst = max(core_err)
print(f"\nworst core relative error : {worst:.2e} (tol. {TOL_U:.0e})")
assert worst < TOL_U, f"velocity off the Hartmann profile by {worst:.3e}"
assert step < STATIONARY, f"not stationary: last step {step:.3e}"
print("OK: the solved velocity IS the Hartmann profile (two-way feedback live).")


In [ ]:
# 3) the two-way discriminator: Hartmann, not Poiseuille.
#    Without the Lorentz feedback the flow would be Poiseuille with a
#    centreline velocity cosh(Ha)(cosh(Ha)-1)^-1 ~ 12.7 times larger; the
#    simulated profile must sit FAR closer to Hartmann than to Poiseuille.
TOL_DISCRIM = 5e-2
worst_ratio = 0.0
for name, y in CORE:
    ux = read_son_comps(case.dataFileName_, name)[0]
    d_h = abs(ux[-1] - u_hartmann(y))     # distance to Hartmann
    d_p = abs(u_poiseuille(y) - u_hartmann(y))   # discriminator distance
    worst_ratio = max(worst_ratio, d_h / d_p)
print(f"max |u - u_Hartmann| / |u_Poiseuille - u_Hartmann| = {worst_ratio:.2e} (tol. {TOL_DISCRIM:.0e})")
print(f"Poiseuille centreline / Hartmann centreline = {(G*H*H/(2*MU))/U0c:.1f} (the suppression factor)")
assert worst_ratio < TOL_DISCRIM, f"the profile leans Poiseuille: ratio {worst_ratio:.3e}"
print("OK: the Lorentz feedback suppresses the flow exactly as the Hartmann solution demands.")


In [ ]:
# 4) the induced field generated by the SOLVED velocity matches the coupled
#    analytic profile (the stretching source acts on the solved u)
TOL_B = 1e-2
for name, y in CORE_B:
    t, b = read_son(case.dataFileName_, name)
    ref = b_x_steady(y)
    rel = abs(b[-1] - ref) / (abs(ref) + 1e-12)
    print(f"{name}  y = {y:+.7f} : b_x = {b[-1]:+.6e}  (analytic {ref:+.6e})  rel = {rel:.2e}")
    assert rel < TOL_B, f"induced field off by {rel:.3e} at {name}"
print("OK: b_x is the analytic induced profile of the coupled steady state.")


In [ ]:
# 5) the magnetic-pressure boundary profiles leave no cross-flow, and the
#    wall-adjacent velocity cell stays inside the documented bound (the
#    grad-u wall bias of the stretching source - see Hartmann1D / the
#    standalone notebooks).
TOL_UY = 1e-4
TOL_WALL = 2e-2
worst_uy = 0.0
for name, y in CORE:
    uy = read_son_comps(case.dataFileName_, name)[1]
    worst_uy = max(worst_uy, abs(uy[-1]))
print(f"cross-flow: max |u_y| at the core probes = {worst_uy:.2e} (bound {TOL_UY:.0e})")
assert worst_uy < TOL_UY, f"cross-flow u_y = {worst_uy:.3e}"
print("OK: the analytic magnetic-pressure BC leaves no cross-flow.")

for name, y in WALL:
    ux = read_son_comps(case.dataFileName_, name)[0]
    ref = u_hartmann(y)
    rel = abs(ux[-1] - ref) / abs(ref)
    print(f"{name}  y = {y:+.7f} : u_x = {ux[-1]:.6e}  (Hartmann {ref:.6e})  rel = {rel:.2e}  (bounded)")
    assert rel < TOL_WALL, f"wall-adjacent cell off by {rel:.3e}"
print("OK: the wall-adjacent cell is inside the documented bound.")


In [ ]:
import matplotlib.pyplot as plt

yy = np.linspace(-H, H, 201)
plt.figure(figsize=(6.5, 4.0))
plt.plot(yy, [u_poiseuille(y) for y in yy], "b--", lw=1.5, label=r"Poiseuille (no feedback): $u_P(y)$")
plt.plot(yy, [u_hartmann(y) for y in yy], "k-", lw=2, label=r"Hartmann (coupled steady): $u(y)$")
for name, y in (CORE + WALL):
    ux = read_son_comps(case.dataFileName_, name)[0]
    plt.plot([y], [ux[-1]], "o", label=f"{name} ({y:+.3f})")
plt.xlabel("y"); plt.ylabel(r"$u_x$")
plt.title("HartmannTwoWay: the Lorentz feedback suppresses the flow by " + f"{(G*H*H/(2*MU))/U0c:.1f}$\times$")
plt.grid(True, ls=":"); plt.legend(loc="best", fontsize=8)
plt.show()


## Known limitation: the velocity gradient at wall cells

The M1 source needs the cell gradient $\nabla u$ of the solved velocity.
The kernel builds it (`Discret_Thyd::grad_u` →
`grad_Champ_Face_PolyMAC_MPFA`) by *averaging the face gradients*
(`update_ge()`), and a boundary face only receives a Dirichlet
contribution — the two cells adjacent to a wall therefore receive a biased
$\nabla u$ (the same documented O(h) limitation as the standalone
`Hartmann1D` form; see `KERNEL_REUSE_NOTES.md`). The wall-adjacent probe is
checked against a bound only; the core probes are asserted tightly.


## Mesh convergence

The shared refinement ladder keeps the streamwise mesh fixed and refines the
active $y$ direction (`coarse` = the baseline mesh `5×65`, `standard` = y
cells ×2 (`5×129`), `fine` = y cells ×4 (`5×257`)). Probes remain in the same
streamwise cell. The mesh-sensitivity metric below compares the *normalized
profile shape* to the analytic Hartmann shape, so a small amplitude offset
common to all probes does not mask spatial convergence.

The effective cell-centre positions are read from each `.son` header; the
observed order is printed as $\log_2(e_k/e_{k+1})$.


In [ ]:
CORE_SCAN = [tag for tag, _ in CORE]     # analytic y is taken from the .son header

def probe_xy(path):
    """Effective (x, y) of a moved point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2])
    raise RuntimeError(f"no probe position header in {path}")

def read_son_dir(subdir, case_name, name):
    """(time, values, (x_eff, y_eff)) of probe `name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float), probe_xy(path)

def core_error(subdir, case_name):
    """Max normalized-profile error, cancelling a common streamwise offset."""
    _, u0, (_, y0) = read_son_dir(subdir, case_name, CORE_SCAN[0])
    shape0 = u_hartmann(y0) / U0c
    worst = 0.0
    for tag in CORE_SCAN[1:]:
        _, u, (_, y_eff) = read_son_dir(subdir, case_name, tag)
        measured_shape = u[-1] / u0[-1]
        exact_shape = (u_hartmann(y_eff) / U0c) / shape0
        worst = max(worst, abs(measured_shape - exact_shape))
    return worst

def by_drift(subdir, case_name):
    """max |b_y(t) - B0| over time at the applied-field probe."""
    _, by = read_son_dir(subdir, case_name, "SONDE_BY")[:2]
    return float(np.max(np.abs(by - B0)))

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (5, 65), "standard": (5, 129), "fine": (5, 257)}
scan_err = {lvl: core_error(sub, cn) for lvl, (sub, cn) in scan.items()}
scan_drift = {lvl: by_drift(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes     profile-shape err observed order   by drift")
prev = None
for lvl in ("coarse", "standard", "fine"):
    nx, ny = scan_nodes[lvl]
    e = scan_err[lvl]
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<9} {nx:>2}x{ny:<4} {e:<16.4e} {order:<16} {scan_drift[lvl]:.2e}")
    assert scan_drift[lvl] < TOL_BY, f"applied field drift at {lvl}: {scan_drift[lvl]}"
    prev = e

assert scan_err["standard"] < scan_err["coarse"], f"no convergence coarse -> standard: {scan_err}"
assert scan_err["fine"] < scan_err["standard"], f"no convergence standard -> fine: {scan_err}"
print("OK: the velocity error decreases monotonically; the b_y invariant holds at every level.")


## Hartmann-number scan (regime map)

The same two-way Hartmann case at Ha = 2.5 and 7.9 (the mesh-scan baseline
above is Ha = 5): eta is set from Ha (eta = (B0 h/Ha)^2/(mu mu0)) and the
pressure profiles follow, so every level has its own exact steady state.
The regime-map quantity is the **suppression factor**

$$ \sigma(\mathrm{Ha}) = \frac{u_P(0)}{u_H(0)} = \frac{\mathrm{Ha}^2}{2\left(1 - 1/\cosh(\mathrm{Ha})\right)}, $$

the ratio of the Poiseuille centreline velocity to the Hartmann one — the
low-Rm interaction strength. The measured profile must reproduce it at
every Ha.


In [ ]:
# 6) regime map: the suppression factor sigma(Ha) at Ha = 2.5, 5, 7.9
TOL_HA = 2.5e-2
TOL_SIGMA = 3e-2

def read_son_sub(sub, cname, name):
    # values of probe `name` in a build subdirectory (per-Ha cases)
    path = f"{BUILD}/{sub}/{cname}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(v, dtype=float)

print(f"{'Ha':>5} {'u_sim(0)':>12} {'u_H(0)':>12} {'rel err':>9} {'sigma_sim':>10} {'sigma_th':>9} {'b_x rel':>9} {'b_y drift':>10}")
has = []
sigs_sim, sigs_th = [], []
for ha in (2.5, 5.0, 7.9):
    if ha == 5.0:                    # the mesh-scan baseline (build/coarse)
        ux = read_son_comps(case.dataFileName_, "SONDE_U0")[0]
        _, b0 = read_son(case.dataFileName_, "SONDE_B0")
        _, by = read_son(case.dataFileName_, "SONDE_BY")
    else:
        cname = cases[f"ha_{ha}"].dataFileName_
        ux = read_son_sub(f"ha_{ha}", cname, "SONDE_U0")
        b0 = read_son_sub(f"ha_{ha}", cname, "SONDE_B0")
        by = read_son_sub(f"ha_{ha}", cname, "SONDE_BY")
    ch = math.cosh(ha); C = 0.25/ha**2
    uH = C*(1 - 1/ch)
    sig_th = ha**2/(2*(1 - 1/ch))
    sig_sim = (G*H*H/(2*MU))/ux[-1]
    bx_ref = math.sinh(ha*0.0078125/H)*H/(ha*ch) - 0.0078125
    err_u = abs(ux[-1] - uH)/uH
    err_sig = abs(sig_sim - sig_th)/sig_th
    err_b = abs(b0[-1] - bx_ref)/abs(bx_ref)
    drift_by = float(np.max(np.abs(by - B0)))
    print(f"{ha:>5} {ux[-1]:>12.6e} {uH:>12.6e} {err_u:>9.2e} {sig_sim:>10.3f} {sig_th:>9.3f} {err_b:>9.2e} {drift_by:>10.2e}")
    assert err_u < TOL_HA, f"u off at Ha={ha}: {err_u:.3e}"
    assert err_sig < TOL_SIGMA, f"suppression factor off at Ha={ha}: {err_sig:.3e}"
    assert err_b < TOL_HA, f"b_x off at Ha={ha}: {err_b:.3e}"
    assert drift_by < TOL_HA, f"b_y drift at Ha={ha}: {drift_by:.3e}"
    has.append(ha); sigs_sim.append(sig_sim); sigs_th.append(sig_th)
print("OK: the two-way regime map reproduces sigma(Ha) = Ha^2/(2(1 - 1/cosh Ha)) at every point.")


## Reynolds–Hartmann map: $Re_H \times Ha$

The complete 12-point grid scans $Re_H \in \{0.01,0.1,1,10\}$ against
$Ha \in \{2.5,5,7.9\}$ at fixed low $Rm=0.1$. Here
$Re_H = \rho U_H(0)L/\mu$, with $L=1$ and $U_H(0)$ the Hartmann centreline
speed. For each pair, viscosity sets $Re_H$, magnetic diffusivity is chosen to
hold $Rm$, and the applied field is set to hold $Ha$. The checks include the
measured $Re_H$ and $Rm$, analytic velocity and induced-field probes, the
suppression factor, and the applied-field invariant.

**Interpretation limit:** this fully developed channel has $u=(u_x(y),0)$, so
$u\cdot\nabla u=0$. Its normalized Hartmann profile and suppression factor
are therefore independent of $Re_H$. The scan covers the parameter plane and
checks dimensional scaling, but does **not** validate an inertial Re transition
or a general $M=Ha/\sqrt{Re}$ regime map. That needs a geometry with nonzero
convective acceleration (for example, a duct entrance). An exploratory
$Re_H=100, Ha=7.9$ run left the analytic Hartmann branch in this finite duct;
that higher-Re region is not included in the checked map.


In [ ]:
# Assert all points in the full Re_H x Ha matrix at fixed low Rm=0.1.
TOL_REHA_U = 2.5e-2
TOL_REHA_B = 2.5e-2
TOL_REHA_SIGMA = 3.0e-2
TOL_REHA_STATIONARY = 1.0e-4
TOL_CROSSFLOW_REL = 1.0e-3
reha_sigma = np.zeros((len(RE_VALUES), len(HA_VALUES)))
interaction_map = np.zeros_like(reha_sigma)
reha_rows = []

for i, re_target in enumerate(RE_VALUES):
    for j, ha in enumerate(HA_VALUES):
        sub = f"re_{re_target:g}_ha_{ha:g}"
        cname = cases[sub].dataFileName_
        centre_shape = 1.0 - 1.0 / math.cosh(ha)
        mu = math.sqrt(RHO_MAP*G_MAP*H_MAP**2*L_MAP*centre_shape /
                       (re_target*ha**2))
        u_center_ref = G_MAP*H_MAP**2/(mu*ha**2)*centre_shape
        eta = u_center_ref*L_MAP/RM_TARGET
        b0 = ha*math.sqrt(mu*MU0_MAP*eta)/H_MAP
        C = G_MAP*H_MAP**2/(mu*ha**2)
        bscale = b0/eta*C

        u_errors = []
        uy_max = 0.0
        step_u = None
        for name, y in CORE:
            u_components = read_son_comps(cname, name, sub=sub)
            ux = u_components[0]
            uy_max = max(uy_max, abs(u_components[1][-1]))
            u_ref = C * (1.0 - math.cosh(ha*y/H_MAP)/math.cosh(ha))
            u_errors.append(abs(ux[-1] - u_ref) / max(abs(u_center_ref), 1e-12))
            if name == "SONDE_U0":
                step_u = abs(ux[-1] - ux[-2]) / max(abs(u_center_ref), 1e-12)
        err_u = max(u_errors)
        assert step_u < TOL_REHA_STATIONARY, \
            f"velocity not stationary at Re={re_target}, Ha={ha}: {step_u:.3e}"
        uy_rel = uy_max / max(abs(u_center_ref), 1e-12)
        assert uy_rel < TOL_CROSSFLOW_REL, \
            f"relative cross-flow at Re={re_target}, Ha={ha}: {uy_rel:.3e}"
        assert err_u < TOL_REHA_U, f"u profile off at Re={re_target}, Ha={ha}: {err_u:.3e}"

        b_y = -0.3046875
        bx = read_son_sub(sub, cname, "SONDE_B3")
        b_ref = bscale * (math.sinh(ha*b_y/H_MAP)*H_MAP/(ha*math.cosh(ha)) - b_y)
        err_b = abs(bx[-1] - b_ref) / max(abs(b_ref), 1e-12)
        step_b = abs(bx[-1] - bx[-2]) / max(abs(b_ref), 1e-12)
        assert step_b < TOL_REHA_STATIONARY, \
            f"b_x not stationary at Re={re_target}, Ha={ha}: {step_b:.3e}"
        assert err_b < TOL_REHA_B, f"b_x off at Re={re_target}, Ha={ha}: {err_b:.3e}"

        ux0 = read_son_comps(cname, "SONDE_U0", sub=sub)[0][-1]
        y0 = CORE[0][1]
        shape_y0 = 1.0 - math.cosh(ha*y0/H_MAP)/math.cosh(ha)
        u_center_sim = ux0 / shape_y0 * centre_shape
        re_sim = RHO_MAP * u_center_sim * L_MAP / mu
        err_re = abs(re_sim - re_target) / re_target
        assert err_re < TOL_REHA_U, f"Re mismatch at Re={re_target}, Ha={ha}: {err_re:.3e}"
        rm_sim = u_center_sim * L_MAP / eta
        assert abs(rm_sim - RM_TARGET) / RM_TARGET < TOL_REHA_U, \
            f"Rm mismatch at Re={re_target}, Ha={ha}: {rm_sim:.4g}"

        u_p = G_MAP * H_MAP**2 / (2.0*mu)
        sigma_sim = u_p / u_center_sim
        sigma_ref = ha**2 / (2.0*centre_shape)
        err_sigma = abs(sigma_sim - sigma_ref) / sigma_ref
        assert err_sigma < TOL_REHA_SIGMA, f"sigma off at Re={re_target}, Ha={ha}: {err_sigma:.3e}"

        by = read_son_sub(sub, cname, "SONDE_BY")
        by_drift = float(np.max(np.abs(by - b0)) / abs(b0))
        assert by_drift < TOL_BY, f"relative b_y drift at Re={re_target}, Ha={ha}: {by_drift:.3e}"

        reha_sigma[i, j] = sigma_sim
        interaction = ha / math.sqrt(re_target)
        interaction_map[i, j] = interaction
        reha_rows.append((re_target, ha, interaction, re_sim, rm_sim,
                          sigma_sim, sigma_ref, err_u, err_b, by_drift, uy_rel))

print(f"{'Re_H':>9} {'Ha':>5} {'M':>9} {'Re sim':>10} {'Rm sim':>9} {'sigma sim':>11} {'sigma exact':>12} {'u err':>9} {'b_x err':>9} {'b_y drift':>11} {'u_y/U':>9}")
for row in reha_rows:
    print(f"{row[0]:9.4g} {row[1]:5.1f} {row[2]:9.3g} {row[3]:10.4g} {row[4]:9.3g} {row[5]:11.4f} {row[6]:12.4f} {row[7]:9.2e} {row[8]:9.2e} {row[9]:11.2e} {row[10]:9.2e}")
print(f"OK: all {len(reha_rows)} Re_H x Ha points satisfy the exact Hartmann checks at Rm={RM_TARGET:g}.")


## High-Rm check (G = 2, Rm = 2)

The Ha scan above keeps `C/η = 1` (Rm = 1). This cell adds a **high-Rm**
case (`G = 2`, so `Rm = 2`, `Re = 0.02`) at the same Ha = 5. The same
analytic Hartmann reference applies with `C = G h²/(μ Ha²) = 0.02`; the
suppression factor `σ = Poiseuille/Hartmann ≈ 12.5` is unchanged (it depends
only on Ha). The lagged coupling is expected to remain stable at this Rm.


In [ ]:
# High-Rm check: G = 2 (Rm = 2) at Ha = 5
G_hr = 2.0
Ha_hr = 5.0
H_hr = 0.5
ETA_hr = 0.01
C_hr = G_hr * H_hr**2 / (1.0 * Ha_hr**2)   # = 0.02
CH_hr = math.cosh(Ha_hr)
U0c_hr = C_hr * (1 - 1/CH_hr)

def u_hartmann_hr(y):
    return C_hr * (1 - math.cosh(Ha_hr * y / H_hr) / CH_hr)

def b_x_steady_hr(y):
    return (1.0 / ETA_hr) * C_hr * (math.sinh(Ha_hr * y / H_hr) * H_hr / (Ha_hr * CH_hr) - y)

# The high-Rm dataset was run as part of the main run cell (cell 6);
# its case name is "PAR_hartmann_two_way_highrm" (partitioned, 2 ranks).
cname = "PAR_hartmann_two_way_highrm"
sub = "hr"

# 1) b_y invariant
t, by = read_son(cname, "SONDE_BY", sub=sub)
drift_by = float(np.max(np.abs(by - 1.0)))
print(f"high-Rm b_y drift: {drift_by:.2e}")
assert drift_by < 3e-2, f"b_y drifted: {drift_by:.3e}"

# 2) velocity relaxes to Hartmann
ux = read_son_comps(cname, "SONDE_U0", sub=sub)[0]
ref_u = u_hartmann_hr(0.0078125)
rel_u = abs(ux[-1] - ref_u) / abs(ref_u)
print(f"high-Rm u_x(0) = {ux[-1]:.6e}  (Hartmann {ref_u:.6e})  rel = {rel_u:.2e}")
assert rel_u < 5e-2, f"u off: {rel_u:.3e}"

# 3) induced field matches
_, bx = read_son(cname, "SONDE_B0", sub=sub)
ref_b = b_x_steady_hr(0.0078125)
rel_b = abs(bx[-1] - ref_b) / (abs(ref_b) + 1e-12)
print(f"high-Rm b_x(0.0078) = {bx[-1]:.6e}  (ref {ref_b:.6e})  rel = {rel_b:.2e}")
assert rel_b < 5e-2, f"b_x off: {rel_b:.3e}"

# 4) discriminator: closer to Hartmann than Poiseuille
ux_all = read_son_comps(cname, "SONDE_U0", sub=sub)[0]
uP = (1.0/(2*1.0))*(0.5**2 - 0.0078125**2)
d_h = abs(ux_all[-1] - ref_u)
d_p = abs(uP - ref_u)
print(f"high-Rm discriminator: d_Hartmann = {d_h:.2e}, d_Poiseuille = {d_p:.2e}, ratio = {d_h/d_p:.3e}")
assert d_h / d_p < 5e-2, f"not Hartmann-like: ratio {d_h/d_p:.3e}"

print("OK: high-Rm (Rm=2) case passes all assertions.")


In [ ]:
import matplotlib.pyplot as plt

hax = np.linspace(2.0, 8.5, 120)
plt.figure(figsize=(6.5, 4.0))
plt.plot(hax, [h*h/(2*(1 - 1/math.cosh(h))) for h in hax], "k--", lw=2,
         label=r"analytic $\sigma(\mathrm{Ha}) = \mathrm{Ha}^2/(2(1-1/\cosh \mathrm{Ha}))$")
plt.plot(has, sigs_sim, "o", ms=8, label="simulated (two-way HartmannTwoWay)")
plt.xlabel(r"Hartmann number $\mathrm{Ha}$"); plt.ylabel(r"suppression $\sigma = u_P(0)/u_H(0)$")
plt.title("Regime map: the Lorentz feedback suppresses the flow as $\mathrm{Ha}^2$")
plt.grid(True, ls=":"); plt.legend(fontsize=9)
plt.show()


In [ ]:
# Full Re_H x Ha map: interaction parameter M and measured Hartmann suppression.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), constrained_layout=True)
for ax, values, title, label, fmt in (
        (axes[0], interaction_map, r"Interaction parameter $M=Ha/\sqrt{Re_H}$", r"$M$", ".2g"),
        (axes[1], reha_sigma, r"Measured suppression $\sigma=u_P(0)/u_H(0)$", r"$\sigma$", ".2f")):
    mesh = ax.pcolormesh(HA_VALUES, RE_VALUES, values, shading="nearest", cmap="viridis")
    for i, re_target in enumerate(RE_VALUES):
        for j, ha in enumerate(HA_VALUES):
            ax.text(ha, re_target, format(values[i, j], fmt), ha="center", va="center", color="white")
    ax.set_yscale("log")
    ax.set_xlabel(r"Hartmann number $Ha$")
    ax.set_ylabel(r"Hartmann Reynolds number $Re_H$")
    ax.set_title(title)
    fig.colorbar(mesh, ax=ax, label=label)
    ax.grid(True, which="both", ls=":", alpha=0.35)
plt.show()


## Computer Performance


In [ ]:
run.tablePerf()
